# RPDA8412w: end-to-end run of the framework (synthetic data)

Run the cells from top to bottom, **from a Jupyter started in the repository folder** (the one that contains
`run_all.py`). Nothing here depends on a Windows path or on where the repository lives.

All data are synthetic and fictional. Scores show that the pipeline works; they are not estimates of real-world
performance. The notebook is a thin driver around `run_all.py`, so the notebook and the command line always do
the same thing.

In [ ]:
import subprocess, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
assert (ROOT / "run_all.py").exists(), f"Start Jupyter in the repository folder (run_all.py not found in {ROOT})"

PRETRAINED = False   # True downloads ImageNet weights for the CNN (needs internet; usually better results)

def run(*args):
    cmd = [sys.executable, str(ROOT / "run_all.py"), *args]
    p = subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    assert p.wait() == 0, "step failed, see the output above"

print("Repository:", ROOT)

## 0. Install requirements (once)

In [ ]:
# %pip install -r requirements.txt

## 1. Visual document CNN
Renders 400 fictional genuine documents, splits the **documents** into train/val/test, tampers each split only from
its own documents (so no document or splice donor is shared), then trains and evaluates the CNN.

In [ ]:
run("--steps", "docs", "cnn", *(["--pretrained"] if PRETRAINED else []))

In [ ]:
# Visual check: one genuine page and its three tampered variants from the test split
import csv, matplotlib.pyplot as plt
from PIL import Image
rows = list(csv.DictReader(open(ROOT / "data_manifests" / "cnn" / "test.csv")))
src = next(r["source_file"] for r in rows if r["label"] == "genuine")
shots = sorted([r for r in rows if r["source_file"] == src], key=lambda r: r["tamper_type"])
fig, ax = plt.subplots(1, len(shots), figsize=(4 * len(shots), 6))
for a, r in zip(ax, shots):
    a.imshow(Image.open(ROOT / r["filepath"])); a.set_title(r["tamper_type"]); a.axis("off")
plt.show()

## 2. Metadata / OCR / font Random Forest

In [ ]:
run("--steps", "rf")

## 3. Siamese signature network
Identities are split first, so the test set contains writers the model never saw during training.

In [ ]:
run("--steps", "siamese")

## 4. Fusion meta-classifier with McNemar's test and SHAP
Matched cases are built from the three branches' **held-out** predictions only (`build_matched_cases.py`); the
fusion model is then trained, evaluated (McNemar + SHAP) and cross-validated against simple baselines.

In [ ]:
run("--steps", "fusion")

## 5. Summary (copy into the report)

In [ ]:
run("--steps", "summary")
from IPython.display import Markdown
Markdown((ROOT / "results" / "summary.md").read_text())